In [3]:
'''
The day_trip_agent is a simple but powerful assistant. We're making it a little smarter by teaching it to understand budget constraints.

Agent: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
Session: The conversation history. For this simple agent, it's just a container for a single request-response.
Runner: The engine that connects the Agent and the Session to process your request and get a response.
'''
import os
import sys
import  json
import asyncio
import random
import string
from uuid import uuid4
from typing import List,Any
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display

#----------ADK , Agent and Evaluation components imports here------------------

from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content , Part
from dotenv import load_dotenv
print(" All libraries are imported!")


 All libraries are imported!


In [4]:
load_dotenv()

True

In [5]:
#agent definition
#day_trip_agent
'''
budget constraints :
Agent: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
Session: The conversation history. For this simple agent, it's just a container for a single request-response.
Runner: The engine that connects the Agent and the Session to process your request and get a response.
'''

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name = "day_trip_agent",
        model = "gemini-3.5-flash",
        description = "An AI-powered travel specializing in real-time, personalized itinerary generation. By analyzing a user's real-time mood, specific interests, and spending limits, it instantly builds cohesive, full-day schedules that balance hidden gems with local favorites.",
        instruction = """
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

In [6]:
day_trip_agent = create_day_trip_agent()
print(f" Agent '{day_trip_agent.name}' is created and ready for adventure!")

 Agent 'day_trip_agent' is created and ready for adventure!


In [18]:
#Runner to Help run the agent: This is a HELPER function
async def run_agent_query(agent:Agent,query:str,session : Session,user_id: str,is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n Running query for agent: '{agent.name}' in session: '{session.id}'...")
    runner = Runner(
        agent = agent,
        session_service = session_service,
        app_name = agent.name)
    final_response = ""
    try:
        async for event in runner.run_async(user_id = user_id ,session_id = session.id,new_message = Content(parts=[Part(text = query)],role ="user")):
            if not is_router:
                # Let's see what the agent is thinking! through events 
                print(f"EVENT:{event}")
                if event.is_final_response():
                    final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
        print("\n" + "-"*50)
        print("✅ Final Response:")
        display(Markdown(final_response))
        print("-"*50 + "\n")
    return final_response

In [19]:
# --- Initializing Session Service ---
session_service = InMemorySessionService()
my_user_id = "adk_user_001"

In [20]:
# --- Testing the Day Trip Agent! ---
async def test_run_day_trip_agent():
    #Creating a new single use session for the query
    day_trip_session = await session_service.create_session(app_name = day_trip_agent.name,user_id = my_user_id)
    # new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Jaipur, Rajasthan India. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(agent =day_trip_agent,query = query,session = day_trip_session,user_id = my_user_id)

In [22]:
await test_run_day_trip_agent()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Jaipur, Rajasthan India. Keep it affordable!'

 Running query for agent: 'day_trip_agent' in session: 'b31be5c2-2244-4086-b27c-2180a78d9088'...


Node execution failed with exception
Traceback (most recent call last):
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\google\adk\models\google_llm.py", line 288, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\google\genai\models.py", line 8790, in generate_content
    response = await self._generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\google\genai\models.py", line 7235, in _generate_content
    response = await self._api_client.async_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\google\genai\_api_client.py", line 1793, in async_request
    result = await self._async_request(
             ^^^^^^^^^^^^^^^^^^^^^^

EVENT:model_version=None content=None grounding_metadata=None partial=None turn_complete=None turn_complete_reason=None finish_reason=None error_code='_ResourceExhaustedError' error_message="\nOn how to mitigate this issue, please refer to:\n\nhttps://google.github.io/adk-docs/agents/models/google-gemini/#error-code-429-resource_exhausted\n\n\n429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}" interrupted=None custom_metadata=None usage_metadata=None live_session_resumption_update=None live_session_id=None go_away=None 

An error occurred: 'NoneType' object has no attribute 'parts'

--------------------------------------------------



Root node day_trip_agent was cancelled.
Failed to detach context
Traceback (most recent call last):
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\opentelemetry\trace\__init__.py", line 608, in use_span
    yield span
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\opentelemetry\trace\__init__.py", line 508, in start_as_current_span
    yield span
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\opentelemetry\trace\__init__.py", line 443, in start_as_current_span
    yield span
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\google\adk\telemetry\_instrumentation.py", line 78, in record_invocation
    yield
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\google\adk\runners.py", line 699, in _run_node_async
    yield event
GeneratorExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\Lenovo\AppData\Roa